# 04 - Real Data Ingestion + Hybrid Pipeline

This notebook connects real-world data into Sentinel pipeline:

- KEV + EPSS -> `VULNERABILITY_EVENT`
- CIC / CAIDA traffic rows -> `DDOS_SIGNAL_EVENT` and `WEB_ATTACK_EVENT`
- Optional synthetic fallback for telco-private classes

Then runs feature build + cyber GNN train.

## Recommended datasets (public / reproducible)
- CISA KEV JSON feed (official)
- FIRST EPSS API (official)
- CIC-IDS2018 (Kaggle mirror) for web/ddos traffic labels
- CAIDA DDoS traces for volumetric normalization

## Data flow into model
raw rows -> `app.integrations.real_data_pipeline` normalizers -> connector events -> `event_log` -> `graph_feature_snapshot` -> GNN train/eval

This is the path that replaces hardcoded-only demos with reproducible external evidence.


## Inputs you provide

- `SOURCE_API_KEY`: use one seeded source key, default `safaricom-secret-key`
- `CIC_INPUT_FILE`: local CSV/JSONL file from CIC dataset
- `CAIDA_INPUT_FILE`: local CSV/JSONL file from CAIDA traces

Tip: keep raw datasets outside git and pass absolute file paths from your machine.


In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()
NOTEBOOKS_DIR = HERE if HERE.name == 'notebooks' else (HERE / 'notebooks')
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

from pipeline_bootstrap import (
    bootstrap_environment,
    check_notebook_prerequisites,
    repair_notebook_schema,
    event_type_counts_last_24h,
    ingest_kev_epss,
    ingest_traffic_file,
    run_feature_snapshots,
    seed_default_sources,
    train_gnn,
)

bootstrap_environment()
pre = check_notebook_prerequisites()
if not pre.get('ok'):
    print(pre.get('hint'))
    print('repair:', repair_notebook_schema())
seed_default_sources()


In [ ]:
SOURCE_API_KEY = 'safaricom-secret-key'
ASSET_ID = 'county-finance-db-01'

RUN_KEV = True
RUN_CIC = False
RUN_CAIDA = False

CIC_INPUT_FILE = '/absolute/path/to/cic_rows.csv'
CAIDA_INPUT_FILE = '/absolute/path/to/caida_rows.csv'


In [ ]:
results = []
if RUN_KEV:
    results.append(ingest_kev_epss(source_api_key=SOURCE_API_KEY, asset_id=ASSET_ID))

if RUN_CIC:
    results.append(
        ingest_traffic_file(
            dataset='cic',
            input_file=CIC_INPUT_FILE,
            source_api_key=SOURCE_API_KEY,
            service_id_prefix='kenya-infra',
            dataset_name='cic_ids2018_kaggle',
        )
    )

if RUN_CAIDA:
    results.append(
        ingest_traffic_file(
            dataset='caida',
            input_file=CAIDA_INPUT_FILE,
            source_api_key=SOURCE_API_KEY,
            service_id_prefix='kenya-infra',
            dataset_name='caida_ddos',
        )
    )

results


In [ ]:
counts = event_type_counts_last_24h()
counts


In [ ]:
feature_stats = run_feature_snapshots(window_keys=('Wmid',), max_entities=6000)
feature_stats


In [ ]:
train_result = train_gnn(domain='cyber', epochs=60)
train_result
